# ClusterMind Chaos Arena — Training Notebook (TRL / Unsloth-compatible)

**This notebook is judge-runnable end-to-end on a free Colab T4.**

Two paths exist; the script picks automatically based on what's installed:

| Path | When | What runs |
|---|---|---|
| **LLM / LoRA (preferred)** | `transformers + peft` available (Colab default) | Qwen2.5-0.5B-Instruct base, **frozen weights**, LoRA r=8, SFT cross-entropy → online RL on live env |
| **Policy-net plumbing (fallback)** | Bare torch only (laptop CPU) | Tiny MLP over engineered features, BC + REINFORCE — proves the loop with no GPU |

> **Disclosure (PRD §26):** We freeze the base model and update only LoRA adapter weights during SFT and GRPO/PPO/REINFORCE training.

Estimated runtime: ~12 min quick mode, ~40 min `--full` on a T4.

## 1 — Install dependencies

In [ ]:
!pip install -q openenv-core pydantic numpy matplotlib pandas plotly networkx PyYAML
!pip install -q transformers accelerate peft trl sentencepiece einops datasets bitsandbytes
!pip install -q gradio
# wandb is optional; uncomment to use:
# !pip install -q wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2 — Clone the repo

In [ ]:
import os
REPO = '/content/clustermind-chaos-arena'
if not os.path.isdir(REPO):
    !git clone --depth=1 https://github.com/Kabeer-Scaler/META_RL_HACKATHON_CLUSTERMIND $REPO || echo 'replace URL with your fork before running'
%cd $REPO
import sys; sys.path.insert(0, REPO)
print('repo ready at', REPO)

## 3 — Imports

In [ ]:
from clustermind import ClusterMindChaosEnv, ClusterMindEnv
from clustermind.baselines import ALL_BASELINES, RandomAgent, GreedyThroughputAgent, ThermalAwareHeuristicAgent
from clustermind.agents import LLMJsonAgent, TransformersBackend
from clustermind.scenarios import SCENARIO_NAMES
print('baselines:', list(ALL_BASELINES.keys()))
print('scenarios:', SCENARIO_NAMES)

## 4 — Smoke tests (17 checks must pass)

In [ ]:
!python scripts/run_smoke_tests.py

## 5 — Baseline quick sweep
Runs all 5 heuristics on all 8 scenarios at curriculum levels 3/4/5. Result is the comparison ground truth that the trained agent has to beat.

In [ ]:
!python scripts/run_baselines.py --episodes 5 --levels 3 4 5 --output results/baseline_metrics.json

In [ ]:
import json
with open('results/baseline_metrics.json') as f:
    payload = json.load(f)
print(f"{'agent':32s} {'reward':>7s} {'crit%':>6s} {'outage':>7s} {'cascade':>8s} {'gv_rate':>8s}")
print('-' * 75)
for name, blob in payload['agents'].items():
    s = blob['summary']
    print(f"{name:32s} {s['avg_reward']:7.2f} {s['critical_completion_rate']*100:5.1f}%"
          f" {s['avg_outage_count']:7.2f} {s['avg_cascade_count']:8.2f} {s['avg_guardrail_violation_rate']:8.2f}")

## 6 — Collect heuristic rollouts (SFT seed data)
Filtered for: valid action AND no guardrail violation AND positive reward (PRD §26.1).

In [ ]:
from scripts.train_trl import collect_heuristic_rollouts
rollouts = collect_heuristic_rollouts(n_episodes=16, seed_base=7000)
print(f'Collected {len(rollouts)} filtered transitions for SFT')
by_action = {}
for r in rollouts:
    by_action[r.abstract_action.value] = by_action.get(r.abstract_action.value, 0) + 1
print('action distribution:', by_action)

## 7 — Train: frozen base + LoRA, SFT → GRPO/PPO/REINFORCE

The script picks the strongest RL algorithm available:
- `--rl-algo grpo` — episode-level group-relative advantage with K=2 trajectories per seed
- `--rl-algo ppo` — REINFORCE + KL penalty against the frozen reference policy
- `--rl-algo reinforce` — moving-baseline REINFORCE (universal fallback)
- `--rl-algo auto` — picks GRPO if `trl` is installed, else REINFORCE

Use `--quick` for ~10-min Colab runs, `--full` for ~40-min runs.

In [ ]:
!python scripts/train_trl.py \
    --mode auto \
    --base-model Qwen/Qwen2.5-0.5B-Instruct \
    --rl-algo auto \
    --grpo-group-size 2 \
    --sft-episodes 16 \
    --rl-episodes 24 \
    --eval-episodes 6 \
    --seed 42 \
    --quick

In [ ]:
import json
with open('results/trained_results.json') as f:
    tr = json.load(f)
print(f"schema:    {tr.get('schema')}")
print(f"rl_algo:   {tr.get('rl_algo')}  ({tr.get('rl_algo_note', '')})")
print(f"frozen_base={tr.get('frozen_base')}  lora_only={tr.get('lora_only')}")
if 'trainable_params' in tr:
    pct = tr['trainable_params'] / max(1, tr['total_params']) * 100
    print(f"trainable: {tr['trainable_params']:,} / {tr['total_params']:,} ({pct:.2f}%)")
if tr.get('rl_rewards'):
    print(f"RL reward: first 5={[round(r,2) for r in tr['rl_rewards'][:5]]}  last 5={[round(r,2) for r in tr['rl_rewards'][-5:]]}")

## 8 — Evaluate: 5 baselines + Base LLM + SFT LoRA + RL LoRA
Pass `--rl-adapter` if Step 7 produced one (LLM path); the evaluator will load it on top of the same base model.

In [ ]:
import os
ADAPTER = 'results/adapters/clustermind_lora'
have_lora = os.path.isdir(ADAPTER)
print(f'LoRA adapter present: {have_lora}')
if have_lora:
    !python scripts/evaluate.py \
        --episodes 5 --levels 3 4 5 \
        --include-llm transformers \
        --base-model Qwen/Qwen2.5-0.5B-Instruct \
        --rl-adapter $ADAPTER \
        --output results/evaluation_metrics.json
else:
    print('Falling back to baseline-only evaluation (Step 7 ran the policy-net plumbing path).')
    !python scripts/evaluate.py --episodes 5 --levels 3 4 5 --output results/evaluation_metrics.json

## 9 — Generate the eight required plots from real logs

In [ ]:
!python scripts/generate_plots.py

In [ ]:
from IPython.display import Image, display, Markdown
captions = {
    'reward_curve.png': 'Per-episode RL reward across live env rollouts.',
    'loss_curve.png': 'SFT cross-entropy + RL policy loss.',
    'outage_comparison.png': 'Average outages per episode by agent.',
    'cascade_count_comparison.png': 'Linked-failure cascades per agent.',
    'critical_job_completion.png': 'Critical-job completion rate.',
    'guardrail_violations.png': 'Guardrail-violation rate.',
    'chaos_survival_score.png': 'Chaos-survival proxy (reward x crit completion).',
    'cluster_health_curve.png': 'Cluster health rolling mean during training.',
}
for name, cap in captions.items():
    display(Markdown(f'**{name}** — {cap}'))
    display(Image(filename=f'results/{name}'))

## 10 — Flight Recorder narrative for the storytelling demo

In [ ]:
!python scripts/export_replay.py --agent GreedyThroughputAgent --scenario triple_crisis --level 4 --seed 7

## 11 — Trained-agent demo loop (LoRA path only)

In [ ]:
import os
if os.path.isdir('results/adapters/clustermind_lora'):
    from clustermind import ClusterMindChaosEnv
    from clustermind.agents import LLMJsonAgent, TransformersBackend
    from clustermind.baselines import ThermalAwareHeuristicAgent
    backend = TransformersBackend(model_name='Qwen/Qwen2.5-0.5B-Instruct',
                                  adapter_path='results/adapters/clustermind_lora')
    agent = LLMJsonAgent(backend=backend, fallback_baseline=ThermalAwareHeuristicAgent(), label='RL-LoRA')
    env = ClusterMindChaosEnv()
    obs = env.reset(seed=99, options={'scenario': 'chaos_arena', 'curriculum_level': 5, 'max_steps': 18})
    total = 0.0
    while not obs.done:
        action = agent.act(obs)
        obs, r, done, info = env.step(action)
        total += r
        chaos = info.get('chaos_action') or '-'
        print(f"step {obs.step:>2}: {action.action_type.value:18s} chaos={chaos:25s} reward={r:+.3f} health={obs.cluster_health:.2f}")
    print(f"\ntotal reward: {total:.2f}\nparse stats: {agent.stats()}")
else:
    print('Skipping — LoRA adapter not present (Step 7 ran the policy-net path).')

## 12 — Final summary table

In [ ]:
import json, os
print('=' * 90)
print('FINAL SUMMARY')
print('=' * 90)
if os.path.isfile('results/trained_results.json'):
    with open('results/trained_results.json') as f: tr = json.load(f)
    print(f"Training schema:   {tr.get('schema')}")
    print(f"RL algo used:      {tr.get('rl_algo')}  ({tr.get('rl_algo_note', '')})")
    print(f"Frozen base:       {tr.get('frozen_base')}")
    print(f"LoRA-only updates: {tr.get('lora_only')}")
    if tr.get('eval_mean_reward') is not None:
        print(f"Trained-agent eval mean reward: {tr['eval_mean_reward']:.2f}")
if os.path.isfile('results/evaluation_metrics.json'):
    with open('results/evaluation_metrics.json') as f: ev = json.load(f)
    print()
    print(f"{'agent':25s} {'reward':>7s} {'crit%':>6s} {'outage':>7s} {'cascade':>8s} {'grade':>6s}")
    for name, blob in ev['agents'].items():
        s = blob['summary']
        band = blob.get('overall_band', '-')
        print(f"{name:25s} {s.get('avg_reward', 0):7.2f} {s.get('critical_completion_rate', 0)*100:5.1f}%"
              f" {s.get('avg_outage_count', 0):7.2f} {s.get('avg_cascade_count', 0):8.2f} {band:>6s}")

## Done
Artifacts to commit / upload to the HF Space:
- `results/adapters/clustermind_lora/` — trained LoRA adapter (frozen base, LoRA only)
- `results/training_logs.jsonl` — per-episode reward + loss + algo
- `results/trained_results.json` — final summary with `rl_algo`, `frozen_base`, `lora_only`
- `results/*.png` — the eight required plots
- `results/replays/*.json` — Flight Recorder rollouts

**Honest disclosure:** if Step 7 ran the policy-net plumbing path (no transformers/peft), the schema in `trained_results.json` will be `clustermind.training.policy_net.v1` — that proves the pipeline runs but is *not* an LLM/LoRA result. To get LLM/LoRA evidence, run this notebook on Colab where transformers + peft are installed.